# SACDpy batch single-image reconstruction

This notebook batch-processes 50-frame TIFF stacks into one SACD image per input. Edit the settings cell, preview the planned filenames, then run the batch cell.

## 1. Setup

Run this notebook from the SACDpy package folder. The local `src/` folder is added to the Python path when the package is not installed.

In [13]:
from pathlib import Path
import sys

repo = Path.cwd()
src = repo / "src"
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

from sacdpy.batch import find_input_files, run_batch_reconstruction, sacdpy_output_path

print(f"Working folder: {repo}")

Working folder: /Users/gmgao/GGscripts/GG-general-GuttmanLab/image_analysis/pipelines/SACDpy


## 2. Settings

These are the only values most users need to edit. With `output_dir=None`, output TIFFs are written next to the input TIFFs.

In [ ]:
input_path = Path("/Volumes/guttman/users/gmgao/Imaging_ProcessedData/SPEN/20260413_ONI-gmgao-SPEN_H3K27ac_dSTORM_SACD/processed-SACD/Aux24h/")
glob_pattern = "*frames_1-50.tif"
output_dir = None
overwrite_outputs = False
show_progress = True

pixel_nm = 117.0
na = 1.45
default_wavelength_nm = 561
wavelength_by_name = {"DAPI": 450, "RL560": 580}

mag = 2
iter1 = 7
iter2 = 8
ac_order = 2
subfactor = 0.8
frames_per_sacd = None  # one SACD image from each full 50-frame stack

# Optional SACDm branches. Defaults reproduce the validated SACDm baseline.
ifbackground = False
backgroundfactor = 2.0
ifregistration = False
ifsparsedecon = False
fidelity = 100.0
tcontinuity = 0.1
sparsity = 1.0
sparse_iterations = 100

## 3. Preview inputs and outputs

Run this cell first to confirm the notebook found the expected files and will write the expected output names.

In [15]:
input_files = find_input_files(input_path, glob_pattern)
if not input_files:
    raise FileNotFoundError(f"No input TIFF files found from {input_path} with pattern {glob_pattern!r}")

print(f"Found {len(input_files)} input TIFF(s):")
for input_file in input_files:
    output_file = sacdpy_output_path(input_file, output_dir=output_dir)
    print(f"{input_file.name} -> {output_file.name}")

Found 12 input TIFF(s):
TXSHA-Aux24h-SPEN_JF549_H3K27ac_DL650-SACD-FOV-1-left_frames_1-50.tif -> TXSHA-Aux24h-SPEN_JF549_H3K27ac_DL650-SACD-FOV-1-SACDpy-left.tif
TXSHA-Aux24h-SPEN_JF549_H3K27ac_DL650-SACD-FOV-1-right_frames_1-50.tif -> TXSHA-Aux24h-SPEN_JF549_H3K27ac_DL650-SACD-FOV-1-SACDpy-right.tif
TXSHA-Aux24h-SPEN_JF549_H3K27ac_DL650-SACD-FOV-2-left_frames_1-50.tif -> TXSHA-Aux24h-SPEN_JF549_H3K27ac_DL650-SACD-FOV-2-SACDpy-left.tif
TXSHA-Aux24h-SPEN_JF549_H3K27ac_DL650-SACD-FOV-2-right_frames_1-50.tif -> TXSHA-Aux24h-SPEN_JF549_H3K27ac_DL650-SACD-FOV-2-SACDpy-right.tif
TXSHA-Aux24h-SPEN_JF549_H3K27ac_DL650-SACD-FOV-3-left_frames_1-50.tif -> TXSHA-Aux24h-SPEN_JF549_H3K27ac_DL650-SACD-FOV-3-SACDpy-left.tif
TXSHA-Aux24h-SPEN_JF549_H3K27ac_DL650-SACD-FOV-3-right_frames_1-50.tif -> TXSHA-Aux24h-SPEN_JF549_H3K27ac_DL650-SACD-FOV-3-SACDpy-right.tif
TXSHA-Aux24h-SPEN_JF549_H3K27ac_DL650-SACD-FOV-4-left_frames_1-50.tif -> TXSHA-Aux24h-SPEN_JF549_H3K27ac_DL650-SACD-FOV-4-SACDpy-left.tif
TXSH

## 4. Run batch reconstruction

This cell writes float32 SACD TIFFs. Existing outputs are skipped unless `overwrite_outputs=True`.

In [16]:
results = run_batch_reconstruction(
    input_files,
    output_dir=output_dir,
    overwrite_outputs=overwrite_outputs,
    pixel_nm=pixel_nm,
    na=na,
    default_wavelength_nm=default_wavelength_nm,
    wavelength_by_name=wavelength_by_name,
    mag=mag,
    iter1=iter1,
    iter2=iter2,
    ac_order=ac_order,
    subfactor=subfactor,
    frames_per_sacd=frames_per_sacd,
    ifbackground=ifbackground,
    backgroundfactor=backgroundfactor,
    ifregistration=ifregistration,
    ifsparsedecon=ifsparsedecon,
    fidelity=fidelity,
    tcontinuity=tcontinuity,
    sparsity=sparsity,
    sparse_iterations=sparse_iterations,
    show_progress=show_progress,
)

for result in results:
    if result.status == "written":
        print(f"Wrote {result.output.name} | {result.input_shape} -> {result.output_shape} | {result.wavelength_nm:g} nm")
    else:
        print(f"Skipped {result.output.name} ({result.status})")

results

Output()

Wrote TXSHA-Aux24h-SPEN_JF549_H3K27ac_DL650-SACD-FOV-1-SACDpy-left.tif | (50, 684, 428) -> (1368, 856) | 561 nm
Wrote TXSHA-Aux24h-SPEN_JF549_H3K27ac_DL650-SACD-FOV-1-SACDpy-right.tif | (50, 684, 428) -> (1368, 856) | 640 nm
Wrote TXSHA-Aux24h-SPEN_JF549_H3K27ac_DL650-SACD-FOV-2-SACDpy-left.tif | (50, 684, 428) -> (1368, 856) | 561 nm
Wrote TXSHA-Aux24h-SPEN_JF549_H3K27ac_DL650-SACD-FOV-2-SACDpy-right.tif | (50, 684, 428) -> (1368, 856) | 640 nm
Wrote TXSHA-Aux24h-SPEN_JF549_H3K27ac_DL650-SACD-FOV-3-SACDpy-left.tif | (50, 684, 428) -> (1368, 856) | 561 nm
Wrote TXSHA-Aux24h-SPEN_JF549_H3K27ac_DL650-SACD-FOV-3-SACDpy-right.tif | (50, 684, 428) -> (1368, 856) | 640 nm
Wrote TXSHA-Aux24h-SPEN_JF549_H3K27ac_DL650-SACD-FOV-4-SACDpy-left.tif | (50, 684, 428) -> (1368, 856) | 561 nm
Wrote TXSHA-Aux24h-SPEN_JF549_H3K27ac_DL650-SACD-FOV-4-SACDpy-right.tif | (50, 684, 428) -> (1368, 856) | 640 nm
Wrote TXSHA-Aux24h-SPEN_JF549_H3K27ac_DL650-SACD-FOV-5-SACDpy-left.tif | (50, 684, 428) -> (1368, 85

[BatchResult(input=PosixPath('/Volumes/guttman/users/gmgao/Imaging_ProcessedData/SPEN/20260413_ONI-gmgao-SPEN_H3K27ac_dSTORM_SACD/processed-SACD/Aux24h/TXSHA-Aux24h-SPEN_JF549_H3K27ac_DL650-SACD-FOV-1-left_frames_1-50.tif'), output=PosixPath('/Volumes/guttman/users/gmgao/Imaging_ProcessedData/SPEN/20260413_ONI-gmgao-SPEN_H3K27ac_dSTORM_SACD/processed-SACD/Aux24h/TXSHA-Aux24h-SPEN_JF549_H3K27ac_DL650-SACD-FOV-1-SACDpy-left.tif'), status='written', input_shape=(50, 684, 428), output_shape=(1368, 856), wavelength_nm=561.0, frames_per_sacd=None),
 BatchResult(input=PosixPath('/Volumes/guttman/users/gmgao/Imaging_ProcessedData/SPEN/20260413_ONI-gmgao-SPEN_H3K27ac_dSTORM_SACD/processed-SACD/Aux24h/TXSHA-Aux24h-SPEN_JF549_H3K27ac_DL650-SACD-FOV-1-right_frames_1-50.tif'), output=PosixPath('/Volumes/guttman/users/gmgao/Imaging_ProcessedData/SPEN/20260413_ONI-gmgao-SPEN_H3K27ac_dSTORM_SACD/processed-SACD/Aux24h/TXSHA-Aux24h-SPEN_JF549_H3K27ac_DL650-SACD-FOV-1-SACDpy-right.tif'), status='written'